#Introducción a Tensor Flow

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf

In [3]:
x1 = tf.constant([1,2,3,4])
x2 = tf.constant([5,6,7,8])

In [4]:
res = tf.multiply(x1, x2)
print(res)

tf.Tensor([ 5 12 21 32], shape=(4,), dtype=int32)


In [5]:
config = tf.compat.v1.ConfigProto(log_device_placement = True)
config = tf.compat.v1.ConfigProto(allow_soft_placement = True)

#Aprendizaje neuronal de las señales de tráfico

In [6]:
import os
import skimage.io as io
import skimage.transform

In [7]:
def load_ml_data(data_directory):
    dirs = [d for d in os.listdir(data_directory)
            if os.path.isdir(os.path.join(data_directory, d))]

    labels = []
    images = []

    for d in dirs:
        label_dir = os.path.join(data_directory, d)
        file_names = [os.path.join(label_dir, f)
                      for f in os.listdir(label_dir)
                      if f.endswith(".ppm")]

        for f in file_names:
            images.append(io.imread(f))
            labels.append(int(d))

    return images, labels

In [8]:
main_dir = '/content/drive/MyDrive/Machine Learning - Python/Datasets - Machine Learning/belgian'
train_dir = os.path.join(main_dir, 'Training')
test_dir = os.path.join(main_dir, 'Testing')

In [ ]:
images, labels = load_ml_data(train_dir)

In [ ]:
len(images)

In [ ]:
images = np.array(images, dtype = object)

In [ ]:
images.size

In [ ]:
images.ndim

In [ ]:
labels = np.array(labels)

In [ ]:
labels.size

In [ ]:
labels.ndim

In [ ]:
len(set(labels))

In [ ]:
images[0]

In [ ]:
images.flags

In [ ]:
images.itemsize

In [ ]:
images.nbytes

In [ ]:
images.nbytes / images.itemsize

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.hist(labels, len(set(labels)))
plt.show()

In [ ]:
import random

In [ ]:
rand_signs = random.sample(range(0, len(images)), 6)
rand_signs

In [ ]:
for i in range(len(rand_signs)):
    temp_in = images[rand_signs[i]]
    plt.subplot(1, 6, i + 1)
    plt.imshow(temp_in)
    plt.axis('off')
    plt.subplots_adjust(wspace = 0.5)
    plt.show()
    print('Forma: {0}, min: {1}, max: {2}'.format(temp_in.shape, temp_in.min(), temp_in.max()))

In [ ]:
unique_labels = set(labels)
plt.figure(figsize = (16, 16))
i = 1

for label in unique_labels:
    temp_in = images[list(labels).index(label)]
    plt.subplot(8, 8, i)
    plt.axis('off')
    plt.title('Label {0} ({1})'.format(label, list(labels).count(label)))
    i += 1
    plt.imshow(temp_in)

plt.show()

In [ ]:
type(labels)

numpy.ndarray

#Modelo de Red Neuronal con Tensor Flow
* las imágenes no son todas del mismo tamaño.
* Hay 62 clases de imágenes (desde la 0 hasta la 61).
* La distribución de señales de tráfico no es uniforme (algunas salen más que otras).

In [ ]:
from skimage import transform

In [ ]:
from skimage.color import rgb2gray

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Definir el modelo
model = Sequential([
    # Primera capa convolucional + MaxPooling
    Conv2D(32, (3, 3), activation='relu', input_shape=(30, 30, 1)),
    MaxPooling2D((2, 2)),

    # Segunda capa convolucional + MaxPooling
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    # Aplanar las características
    Flatten(),

    # Capa densa totalmente conectada
    Dense(128, activation='relu'),
    Dropout(0.5),  # Regularización para evitar overfitting

    # Capa de salida (62 categorías para las señales de tráfico)
    Dense(62, activation='softmax')
])

# Compilar el modelo
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

# Mostrar el resumen del modelo
model.summary()

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 28, 28, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 14, 14, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 12, 12, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 6, 6, 64)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 2304)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         295,040 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 62)                  │           7,998 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 321,854 (1.23 MB)

 Trainable params: 321,854 (1.23 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
images30 = [transform.resize(image, (30, 30)) for image in images]

In [ ]:
test_images, test_labels = load_ml_data(test_dir)
test_images30 = [transform.resize(image, (30, 30)) for image in test_images]
test_images30 = rgb2gray(test_images30)

In [ ]:
# Reshape test_images30 to match the model's expected input shape (30, 30, 1)
test_images30 = np.array(test_images30, dtype=np.float32)  # Convert to NumPy array with dtype=np.float32
test_images30 = test_images30.reshape(-1, 30, 30, 1)  # Reshape with channel dimension

In [ ]:
images30 = np.array(images30)
images30 = rgb2gray(images30)

In [ ]:
images30 = images30 / 255.0
test_images30 = test_images30 / 255.0

In [ ]:
# Dividir los datos en entrenamiento y validación
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(images30, labels, test_size=0.2, random_state=42)

# Entrenar el modelo
history = model.fit(x_train, y_train,
                    validation_data=(x_val, y_val),
                    epochs=20,
                    batch_size=32)

# Evaluar el modelo
test_loss, test_accuracy = model.evaluate(test_images30, test_labels)
print(f"Precisión en el conjunto de prueba: {test_accuracy * 100:.2f}%")

test_loss, test_accuracy = model.evaluate(test_images30, test_labels)
print(f"Precisión en el conjunto de prueba: {test_accuracy * 100:.2f}%")

Epoch 1/20
115/115 ━━━━━━━━━━━━━━━━━━━━ 11s 65ms/step - accuracy: 0.1203 - loss: 3.7208 - val_accuracy: 0.4787 - val_loss: 2.3021
Epoch 2/20
115/115 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - accuracy: 0.4912 - loss: 2.1369 - val_accuracy: 0.6754 - val_loss: 1.3675
Epoch 3/20
115/115 ━━━━━━━━━━━━━━━━━━━━ 6s 42ms/step - accuracy: 0.6505 - loss: 1.4242 - val_accuracy: 0.7486 - val_loss: 0.9867
Epoch 4/20
115/115 ━━━━━━━━━━━━━━━━━━━━ 6s 49ms/step - accuracy: 0.7106 - loss: 1.0973 - val_accuracy: 0.8339 - val_loss: 0.6693
Epoch 5/20
115/115 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - accuracy: 0.7599 - loss: 0.8373 - val_accuracy: 0.8634 - val_loss: 0.5173
Epoch 6/20
115/115 ━━━━━━━━━━━━━━━━━━━━ 6s 44ms/step - accuracy: 0.8042 - loss: 0.7071 - val_accuracy: 0.8743 - val_loss: 0.4462
Epoch 7/20
115/115 ━━━━━━━━━━━━━━━━━━━━ 6s 48ms/step - accuracy: 0.8308 - loss: 0.5950 - val_accuracy: 0.8787 - val_loss: 0.3903
Epoch 8/20
115/115 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 0.8497 - loss: 0.4942 - val_acc

ValueError: Unrecognized data type: x=[[[[1.69647470e-01]
   [1.69019297e-01]
   [1.69444844e-01]
   ...
   [1.17180228e-01]
   [1.23998292e-01]
   [1.31052285e-01]]

  [[1.53342828e-01]
   [1.46967471e-01]
   [1.49975255e-01]
   ...
   [1.21495090e-01]
   [1.11956388e-01]
   [1.31365597e-01]]

  [[1.58813030e-01]
   [1.53975099e-01]
   [1.58918992e-01]
   ...
   [1.60095587e-01]
   [1.53376713e-01]
   [1.11976698e-01]]

  ...

  [[1.85872704e-01]
   [2.00261205e-01]
   [2.66578704e-01]
   ...
   [2.13823944e-01]
   [2.06832156e-01]
   [1.85243383e-01]]

  [[1.78792939e-01]
   [2.36301914e-01]
   [3.54388505e-01]
   ...
   [2.10989565e-01]
   [2.10551575e-01]
   [2.29188904e-01]]

  [[1.81211248e-01]
   [2.02226683e-01]
   [2.41604239e-01]
   ...
   [2.10391149e-01]
   [2.08377853e-01]
   [2.27498502e-01]]]


 [[[2.81955987e-01]
   [2.83083647e-01]
   [5.30661285e-01]
   ...
   [1.90311074e-01]
   [1.66744459e-03]
   [9.65833564e-08]]

  [[2.73823261e-01]
   [2.88549453e-01]
   [5.6896

#Evaluación de la red neuronal.

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

# Seleccionar aleatoriamente 16 muestras
sample_idx = random.sample(range(len(images30)), 16)
sample_images = np.array([images30[i] for i in sample_idx])  # Convertir a array para usar con el modelo
sample_labels = np.array([labels[i] for i in sample_idx])

# Realizar las predicciones
predictions = model.predict(sample_images)  # Utilizamos el modelo directamente

# Obtener el índice de la clase con mayor probabilidad
predicted_labels = np.argmax(predictions, axis=1)

# Visualización de las imágenes, etiquetas reales y predicciones
plt.figure(figsize=(16, 9))
for i in range(len(sample_images)):
    truth = sample_labels[i]
    pred = predicted_labels[i]
    plt.subplot(4, 4, i + 1)
    plt.axis('off')
    plt.imshow(sample_images[i], cmap='gray')
    color = 'green' if truth == pred else 'red'
    plt.text(2, 25, f'Real: {truth}\nPred: {pred}', fontsize=10, color=color)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import tensorflow as tf
from skimage.color import rgb2gray
from skimage.transform import resize

# Redimensionar y convertir imágenes a escala de grises
def preprocess_images(images):
    images30 = [resize(image, (30, 30)) for image in images]
    return np.expand_dims(rgb2gray(images30), axis=-1)  # (n_samples, 30, 30, 1)

# Cargar datos de prueba y preprocesar
test_images, test_labels = load_ml_data(test_dir)
test_images30 = preprocess_images(test_images)

# Modelo simple para clasificación
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(30, 30, 1)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(62, activation=None)  # 62 clases de salida
])

# Compilar el modelo
model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

# Realizar predicciones y calcular precisión
predictions = np.argmax(model.predict(test_images30), axis=1)
accuracy = np.mean(predictions == test_labels) * 100
print(f"Precisión en el conjunto de prueba: {accuracy:.2f}%")



79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Precisión en el conjunto de prueba: 0.52%


In [ ]:
test_images, test_labels = load_ml_data(test_dir)
test_images30 = [transform.resize(image, (30, 30)) for image in test_images]
test_images30 = rgb2gray(test_images30)

predictions = sess.run([predictions], feed_dict = {x: test_images30})[0]
match_count = sum([int(l0 == lp) for l0, lp in zip(test_labels, predictions)])
accuracy = match_count / len(test_labels) * 100
print("Accuracy: {:.3f}".format(accuracy)


SyntaxError: incomplete input (<ipython-input-42-3441e36e7355>, line 8)